In [1]:
import os
os.environ["HUGGINGFACE_API"] = ""
os.environ["GIT_TOKEN"] = ""

In [2]:
import os
from huggingface_hub import login


huggingface_api = os.environ["HUGGINGFACE_API"]
git_token = os.environ["GIT_TOKEN"]

if huggingface_api is None:
    raise RuntimeError("❌ Missing HUGGINGFACE_API in .env")

login(token=huggingface_api)


In [3]:
!git clone https://{git_token}@github.com/BGKhanh/Reasoning-Techniques-on-LLM.git

Cloning into 'Reasoning-Techniques-on-LLM'...
remote: Enumerating objects: 2691, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 2691 (delta 139), reused 158 (delta 78), pack-reused 2456 (from 2)
Receiving objects: 100% (2691/2691), 4.64 MiB | 15.94 MiB/s, done.
Resolving deltas: 100% (1779/1779), done.


In [49]:
%cd Reasoning-Techniques-on-LLM

[Errno 2] No such file or directory: 'Reasoning-Techniques-on-LLM'
/workspace/Reasoning-Techniques-on-LLM/lm-evaluation-harness


In [5]:
!git pull

Already up to date.


In [51]:
# Add --ignore-installed if PyJWT error occurs or use "sudo apt remove python3-jwt"
import sys
!{sys.executable} -m pip install -Uqr requirements.txt --ignore-installed

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nvitop 1.7.0 requires nvidia-ml-py<13.596.0a0,>=11.450.51, but you have nvidia-ml-py 13.610.43 which is incompatible.


In [52]:
!export FLASH_ATTENTION_SKIP_CUDA_BUILD=TRUE

In [53]:
import sys
!{sys.executable} -m pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu130torch2.11-cp312-cp312-linux_x86_64.whl

In [54]:
%cd lm-evaluation-harness
import sys
!{sys.executable} -m pip install -qe ."[api]"

/workspace/Reasoning-Techniques-on-LLM/lm-evaluation-harness


In [55]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

def _find_project_root(start: Path, max_up: int = 5) -> Path:
    cur = start.resolve()
    for _ in range(max_up):
        if (cur / "src" / "prompt_templates").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start  

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

VLLM_MODEL_ID = os.getenv("VLLM_MODEL_ID", "nvidia/Gemma-4-26B-A4B-NVFP4")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))

BASE_URL = os.getenv("BASE_URL", f"http://127.0.0.1:{VLLM_PORT}/v1")
base_url_completions = BASE_URL.rstrip("/") + "/completions"

OUTPUT_DIR = PROJECT_ROOT / "results" / "lm_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = /workspace/Reasoning-Techniques-on-LLM


# Host VLLM

In [ ]:
# Khởi chạy vLLM OpenAI server cục bộ cho student LM (chạy sau cell Configuration).
# Tùy VRAM có thể thêm vào _vllm_cmd: --max-model-len 4096 --gpu-memory-utilization 0.9 --dtype bfloat16
import subprocess
import sys
import time
import urllib.error
import urllib.request

if "_vllm_proc" in globals() and _vllm_proc is not None and _vllm_proc.poll() is None:
    _vllm_proc.terminate()
    try:
        _vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        _vllm_proc.kill()

_vllm_env = os.environ.copy()
_hf = (
    _vllm_env.get("HUGGINGFACE_API_KEY")
    or _vllm_env.get("HF_TOKEN")
    or _vllm_env.get("HUGGINGFACE_HUB_TOKEN")
    or _vllm_env.get("HUGGINGFACE_API", "")
)
if _hf:
    _vllm_env.setdefault("HF_TOKEN", _hf)
    _vllm_env.setdefault("HUGGINGFACE_HUB_TOKEN", _hf)

_vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", VLLM_MODEL_ID,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--dtype", "bfloat16",
    "--gpu-memory-utilization", "0.975",
    # "--tensor-parallel-size", "2",
    "--max-model-len", "16384",
    # "--max-num-batched-tokens", "8192"
]

_vllm_log = OUTPUT_DIR / "vllm_server.log"
_flog = open(_vllm_log, "a", encoding="utf-8", buffering=1)
_flog.write(f"\n\n==== vLLM start {time.strftime('%Y-%m-%d %H:%M:%S')} ====\n")
_flog.write(" ".join(_vllm_cmd) + "\n")
_flog.flush()

print("Starting vLLM — log:", _vllm_log)
_vllm_proc = subprocess.Popen(
    _vllm_cmd,
    env=_vllm_env,
    stdout=_flog,
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
)

_health = BASE_URL.rstrip("/") + "/models"
_deadline = time.time() + float(os.getenv("VLLM_START_TIMEOUT_S", "1200"))
_last_err = None
while time.time() < _deadline:
    if _vllm_proc.poll() is not None:
        _flog.close()
        tail = _vllm_log.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            f"vLLM exited early (code={_vllm_proc.returncode}). See {_vllm_log}. Tail:\n{tail}"
        )
    try:
        with urllib.request.urlopen(_health, timeout=5) as r:
            if r.status == 200:
                print(f"vLLM ready: {_health}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        _last_err = e
    time.sleep(2.0)
else:
    if _vllm_proc.poll() is None:
        _vllm_proc.terminate()
    _flog.close()
    raise RuntimeError(f"vLLM did not become ready in time. Last error: {_last_err}. Log: {_vllm_log}")

In [ ]:
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model local-completions \
  --model_args model=nvidia/Gemma-4-26B-A4B-NVFP4,base_url=http://127.0.0.1:8000/v1/completions,num_concurrent=32,max_length=16384 \
  --tasks vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --output_path results/lm_eval/gemma_few_shot_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":3}' \
  --confirm_run_unsafe_code

# Host llama.cpp

In [ ]:
import sys
!{sys.executable} -m pip install llama-cpp-python==0.3.30 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130

In [ ]:
import sys 
!{sys.executable} -m pip install uvicorn starlette "fastapi<0.137.0" sse_starlette starlette_context pydantic_settings sentencepiece tiktoken

In [ ]:
import subprocess
import sys
import time
import urllib.request
import urllib.error
import os
import atexit
from pathlib import Path

MODEL_PATH = "google/gemma-4-26B-A4B-it-qat-q4_0-gguf"
PORT = 8000
LOG_FILE = Path("llama_server.log")

def cleanup_llama_server():
    global _llama_proc
    if '_llama_proc' in globals() and _llama_proc is not None:
        if _llama_proc.poll() is None:
            print("\n[Cleanup] Đang tắt llama.cpp server an toàn...")
            _llama_proc.terminate()
            try:
                _llama_proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                _llama_proc.kill()

atexit.register(cleanup_llama_server)
cleanup_llama_server()

llama_cmd = [
    sys.executable, "-m", "llama_cpp.server",
    "--hf_model_repo_id", str(MODEL_PATH),
    "--model", "gemma-4-26B_q4_0-it.gguf",
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "--n_gpu_layers", "-1",
    "--n_ctx", "8192",
]

flog = open(LOG_FILE, "w", encoding="utf-8", buffering=1)
flog.write(f"==== Khởi chạy lúc {time.strftime('%H:%M:%S')} ====\n{' '.join(llama_cmd)}\n\n")

print(f"🚀 Đang khởi chạy llama-cpp-python server (Log: {LOG_FILE})...")
_llama_proc = subprocess.Popen(llama_cmd, stdout=flog, stderr=subprocess.STDOUT, env=os.environ.copy())

health_url = f"http://127.0.0.1:{PORT}/v1/models"
deadline = time.time() + 180
last_err = None

while time.time() < deadline:
    if _llama_proc.poll() is not None:
        flog.close()
        tail = LOG_FILE.read_text(encoding="utf-8", errors="replace")[-2000:]
        raise RuntimeError(f"❌ Server bị crash (Exit code: {_llama_proc.returncode}). Log:\n{tail}")
    try:
        with urllib.request.urlopen(health_url, timeout=2) as r:
            if r.status == 200:
                print(f"✅ Sẵn sàng tại: http://127.0.0.1:{PORT}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        last_err = e
    time.sleep(2.0)
else:
    cleanup_llama_server()
    flog.close()
    raise RuntimeError(f"⏳ Quá thời gian chờ. Lỗi cuối: {last_err}. Log: {LOG_FILE}")

In [ ]:
!PYTHONHASHSEED=42 \
CUBLAS_WORKSPACE_CONFIG=:4096:8 \
lm_eval \
  --model local-chat-completions \
  --model_args model=google/gemma-4-26B-A4B-it-qat-q4_0-gguf,tokenizer=google/gemma-4-26B-A4B-it,tokenized_requests=False,base_url=http://127.0.0.1:8000/v1/chat/completions,num_concurrent=32,max_retries=3,max_length=16384 \
  --tasks vietnamese_ssa \
  --apply_chat_template \
  --log_samples \
  --output_path results/lm_eval/gemma_gguf_3shot \
  --metadata '{"technique":"few_shot","language":"vi","n_shot":3}' \
  --confirm_run_unsafe_code